# NB05b — PTSAM Soft-Prompt Eval on Potsdam

**Standalone eval. Run this after uploading `soft_prompts.pt` to a Kaggle dataset.**

**Datasets to attach:**
- `dummyirl/sam3-weights` — SAM3 checkpoint
- `dummyirl/6isprs` — Potsdam GeoTIFF + labels
- `harish77718/ptsam-soft-prompts` — your `soft_prompts.pt`

**To upload soft_prompts.pt:**
1. Go to kaggle.com/datasets -> New Dataset
2. Name: `ptsam-soft-prompts`, upload `soft_prompts.pt`
3. Attach it to this notebook via Settings -> Add Dataset

**Output:** mIoU on val tile `6_15` (compare vs zero-shot baseline 57.8%)

## 1 — Environment setup

In [ ]:
import os

!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda_installer.sh
!bash /tmp/miniconda_installer.sh -b -p /tmp/miniconda

os.environ.pop('PYTHONPATH', None)
os.environ['PATH'] = '/tmp/miniconda/bin:' + os.environ['PATH']

!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!conda --version

In [ ]:
!/tmp/miniconda/bin/conda create -n segearth python=3.10 -y

In [ ]:
!conda run -n segearth pip install torch==2.4.0 torchvision==0.19.0 -q

In [ ]:
!conda run -n segearth pip install openmim -q
!conda run -n segearth mim install 'mmcv==2.2.0' -q
!conda run -n segearth pip install 'mmsegmentation==1.2.2' -q

In [ ]:
%%bash
source /tmp/miniconda/bin/activate segearth
python - << 'EOF'
import pathlib
f = pathlib.Path('/tmp/miniconda/envs/segearth/lib/python3.10/site-packages/mmseg/__init__.py')
f.write_text(f.read_text().replace("MMCV_MAX = '2.2.0'", "MMCV_MAX = '2.3.0'"))
print('Patched MMCV_MAX -> 2.3.0')
EOF
pip install numpy==1.26.4 tifffile -q

## 2 — Clone repo

In [ ]:
import subprocess, os
from pathlib import Path

REPO = Path('/tmp/SegEarth-OV-3')

if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
    print(f'Updated -> {REPO}')
else:
    subprocess.run(
        ['git', 'clone', '--depth=1',
         'https://github.com/HarishDeepak/rg-segearth-ov3', str(REPO)],
        check=True)
    print(f'Cloned -> {REPO}')

os.chdir(REPO)
!conda run -n segearth pip install -r requirements.txt -q

## 3 — Eval on val tile 6_15

Loads `soft_prompts.pt` from the attached dataset and computes mIoU on tile `6_15`.
Compare against zero-shot baseline: **57.8% mIoU** (from literature).

In [ ]:
%%bash
export MPLBACKEND=Agg
export PYTHONUNBUFFERED=1
source /tmp/miniconda/bin/activate segearth
cd /tmp/SegEarth-OV-3

python - << 'PYEOF'
import sys, torch, torch.nn.functional as F
import numpy as np, tifffile, math
from pathlib import Path

sys.stdout.reconfigure(line_buffering=True)

DEVICE       = "cuda"
SOFT_PATH    = Path("/kaggle/input/ptsam-soft-prompts/soft_prompts.pt")
POTSDAM_ROOT = Path("/kaggle/input/datasets/dummyirl/6isprs")
VAL_TILE     = "6_15"
N_CLASSES    = 6
IGNORE_IDX   = 255
CROP_SIZE    = 1024
STRIDE       = 1024

if not SOFT_PATH.exists():
    print(f"soft_prompts.pt not found at {SOFT_PATH}")
    print("Attach dataset harish77718/ptsam-soft-prompts in notebook settings.")
    raise SystemExit(1)

soft_prompts = torch.nn.Parameter(torch.load(str(SOFT_PATH)).to(DEVICE))
print(f"Loaded soft_prompts {soft_prompts.shape}  ({soft_prompts.numel()} params)", flush=True)

from config_local import SAM3_CHECKPOINT
from sam3 import build_sam3_image_model
from sam3.model.data_misc import FindStage

print("Loading SAM3...", flush=True)
model = build_sam3_image_model(
    bpe_path="./sam3/assets/bpe_simple_vocab_16e6.txt.gz",
    checkpoint_path=SAM3_CHECKPOINT, device=DEVICE)
model.eval()
for p in model.parameters():
    p.requires_grad = False
print(f"GPU: {torch.cuda.get_device_name(0)}", flush=True)

find_stage = FindStage(
    img_ids=torch.tensor([0], device=DEVICE, dtype=torch.long),
    text_ids=torch.tensor([0], device=DEVICE, dtype=torch.long),
    input_boxes=None, input_boxes_mask=None, input_boxes_label=None,
    input_points=None, input_points_mask=None,
)

CLASS_NAMES = [
    "impervious surface, road, pavement, paved ground",
    "building, rooftop, structure",
    "low vegetation, grass, shrub, lawn, meadow",
    "tree, forest, canopy",
    "car, vehicle",
    "clutter, background",
]
RGB_TO_IDX = {
    (255,255,255):1, (0,0,255):2, (0,255,255):3,
    (0,255,0):4,     (255,255,0):5, (255,0,0):6
}

def rgb_to_gt(path):
    arr = tifffile.imread(str(path))
    if arr.ndim == 2: return None
    h, w = arr.shape[:2]
    gt = torch.full((h,w), IGNORE_IDX, dtype=torch.int64)
    for (r,g,b), idx in RGB_TO_IDX.items():
        m = (arr[:,:,0]==r)&(arr[:,:,1]==g)&(arr[:,:,2]==b)
        gt[m] = idx - 1
    return gt

from torchvision.transforms import v2
img_transform = v2.Compose([
    v2.ToDtype(torch.uint8, scale=True),
    v2.Resize(size=(1008,1008)),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5]),
])

img_path = POTSDAM_ROOT / f"top_potsdam_{VAL_TILE}_RGB.tif"
lbl_path = POTSDAM_ROOT / f"top_potsdam_{VAL_TILE}_label_noBoundary.tif"
img_arr = tifffile.imread(str(img_path))[:,:,:3]
gt_full  = rgb_to_gt(lbl_path)
H_full, W_full = img_arr.shape[:2]
print(f"Val tile {VAL_TILE}: {W_full}x{H_full}px  valid GT: {(gt_full!=IGNORE_IDX).sum():,}", flush=True)

print("Caching text embeddings...", flush=True)
text_embeds = []
with torch.no_grad():
    for n in CLASS_NAMES:
        text_embeds.append(model.backbone.forward_text([n], device=DEVICE))

pred_full  = torch.zeros(N_CLASSES, H_full, W_full, device=DEVICE)
weight_mat = torch.zeros(H_full, W_full, device=DEVICE)

h_grids = math.ceil((H_full - CROP_SIZE) / STRIDE) + 1
w_grids = math.ceil((W_full - CROP_SIZE) / STRIDE) + 1
total   = h_grids * w_grids
print(f"Sliding window: {h_grids}x{w_grids}={total} crops...", flush=True)

for hi in range(h_grids):
    for wi in range(w_grids):
        y1 = min(hi * STRIDE, H_full - CROP_SIZE)
        x1 = min(wi * STRIDE, W_full - CROP_SIZE)
        y2, x2 = y1 + CROP_SIZE, x1 + CROP_SIZE

        crop  = img_arr[y1:y2, x1:x2]
        img_t = torch.from_numpy(crop).permute(2,0,1).float().unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            bb = model.backbone.forward_image(img_transform(img_t))

        crop_logits = torch.zeros(N_CLASSES, CROP_SIZE, CROP_SIZE, device=DEVICE)

        with torch.no_grad():
            for cls_idx, te in enumerate(text_embeds):
                dummy_geom = model._get_dummy_prompt()   # fresh per class — forward_grounding mutates it
                lang_f = te["language_features"].detach()
                lang_m = te["language_mask"].detach()
                soft   = soft_prompts.expand(-1, 1, -1)
                soft_m = torch.ones(1, soft.shape[0], device=DEVICE, dtype=lang_m.dtype)
                this_bb = dict(bb)
                this_bb["language_features"] = torch.cat([lang_f, soft], dim=0)
                this_bb["language_mask"]     = torch.cat([lang_m, soft_m], dim=1)
                out  = model.forward_grounding(this_bb, find_stage, dummy_geom, None)
                sem  = F.interpolate(out["semantic_seg"], size=(CROP_SIZE, CROP_SIZE),
                                     mode="bilinear", align_corners=False)
                pres = out["presence_logit_dec"].sigmoid()
                crop_logits[cls_idx] = sem.squeeze() * pres.squeeze()

        pred_full[:, y1:y2, x1:x2] += crop_logits
        weight_mat[y1:y2, x1:x2]   += 1.0

        done = hi * w_grids + wi + 1
        if done % 4 == 0 or done == total:
            print(f"  {done}/{total} crops done", flush=True)

pred_full = pred_full / weight_mat.unsqueeze(0)
seg_pred  = pred_full.argmax(0).cpu()

tp = torch.zeros(N_CLASSES); fp = torch.zeros(N_CLASSES); fn = torch.zeros(N_CLASSES)
valid = gt_full != IGNORE_IDX
for c in range(N_CLASSES):
    pc = (seg_pred == c) & valid
    lc = (gt_full  == c) & valid
    tp[c] = (pc & lc).sum().float()
    fp[c] = (pc & ~lc).sum().float()
    fn[c] = (~pc & lc).sum().float()

iou      = tp / (tp + fp + fn + 1e-6)
acc      = tp / (tp + fn + 1e-6)
CLS_NAMES = ["impervious","building","low_veg","tree","car","clutter"]
print("\n=== Val tile 6_15 — PTSAM soft-prompt mIoU ===")
print(f"  {'class':<20} {'IoU':>7}  {'Recall':>7}")
print(f"  {'-'*38}")
for n, iou_v, acc_v in zip(CLS_NAMES, iou, acc):
    print(f"  {n:<20} {iou_v*100:6.2f}%  {acc_v*100:6.2f}%")
print(f"  {'-'*38}")
print(f"  {'mIoU':<20} {iou.mean()*100:6.2f}%")
print(f"  {'mAcc':<20} {acc.mean()*100:6.2f}%")
print(f"\nZero-shot baseline (literature): 57.8% mIoU")
delta = iou.mean().item()*100 - 57.8
print(f"Delta vs baseline: {delta:+.2f}%")
PYEOF